# Assignment 3: Assemble IR and LM for RAG

## Overview
Assemble the BM25 retriever from Assignment 1 and the text generator from Assignment 2 to build a RAG system. Test it with open-form questions on the Cranfield collection. This assignment covers four tasks: building the RAG system, selecting the best generator, understanding how retrieval affects output quality, and diagnosing where the pipeline succeeds or fails. All evaluations in this assignment are qualitative. In your analysis, do not just describe what the model outputs, but engage with why it behaves the way it does and what it reveals about the system.

## Requirements
- Make sure all cells have been run and outputs are visible. Queries and model responses **must be displayed** in the notebook output.
- Implement each task in the cells marked **TODO** and answer questions marked **TODO**

## Deadline
**May 1, 23:59 CET**

## Submission
- Submit only the .ipynb file. Add your name to the filename.

In [12]:
# Load imports
from __future__ import annotations

import math
import re
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Dict, Iterable, List, Tuple, Optional

import numpy as np
import torch
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from transformers import AutoTokenizer, AutoModelForCausalLM
import ir_datasets


## Load the models and tokenizer

In [2]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-0.5B")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_full   = AutoModelForCausalLM.from_pretrained("/home/t-dstetsenko/RAG-Lecture-UZH-SS26/rag/Assignment 2/ckpt-full")

model_completion = AutoModelForCausalLM.from_pretrained("/home/t-dstetsenko/RAG-Lecture-UZH-SS26/rag/Assignment 2/ckpt-completion")

print("Both models loaded.")

Both models loaded.


## Task 1: Build a RAG System

### Task overview
1. Implement a `BM25` class (ported from Assignment 1) and build an index over the Cranfield collection.
2. Implement a `RAGSystem` class that ties retrieval and generation together.




### Task 1.1: BM25 Retriever

Port your `BM25` implementation from Assignment 1, keeping only the attributes and methods you require.

In [4]:
class BM25:
    """
    Minimal BM25 retriever (ported from Assignment 1).

    Brute-force scoring over all documents (no inverted index).
    Only the attributes/methods required by the RAG pipeline are kept:
      - tokenize(text)
      - build(docs_iter)
      - idf(term), score(query_tokens, doc_id)
      - retrieve(query_text, k)
    The ``docs_store`` mapping is kept so that the RAG system can look up
    the raw title/text of each retrieved document.
    """

    def __init__(self, k1: float = 1.2, b: float = 0.75, remove_stopwords: bool = True):
        # BM25 hyperparameters
        self.k1 = k1                # term-frequency saturation
        self.b = b                  # document length normalisation
        self.remove_stopwords = remove_stopwords

        # Tokenization helpers
        self._stemmer = PorterStemmer()
        self._stopwords = set(stopwords.words("english")) if remove_stopwords else set()

        # Corpus statistics (populated by build())
        self.N: int = 0
        self.avgdl: float = 0.0
        self.df: Dict[str, int] = {}
        self.doc_len: Dict[str, int] = {}
        self.doc_tf: Dict[str, Counter] = {}
        self.docs_store: Dict[str, Dict[str, str]] = {}

    # -------------------------- tokenization --------------------------
    def tokenize(self, text: str) -> List[str]:
        text = text.lower()
        tokens = re.findall(r"\b\w+\b", text)
        out = []
        for tok in tokens:
            if tok in self._stopwords:
                continue
            stem = self._stemmer.stem(tok)
            if stem:
                out.append(stem)
        return out

    # -------------------------- index build ---------------------------
    def build(self, docs_iter: Iterable) -> None:
        for doc in docs_iter:
            doc_id = doc.doc_id
            title = doc.title or ""
            text = doc.text or ""
            tokens = self.tokenize(f"{title} {text}")
            tf = Counter(tokens)
            self.doc_tf[doc_id] = tf
            self.doc_len[doc_id] = sum(tf.values())
            self.docs_store[doc_id] = {"title": title, "text": text}

        df_counter = Counter()
        for tf in self.doc_tf.values():
            df_counter.update(set(tf.keys()))
        self.df = dict(df_counter)
        self.N = len(self.doc_tf)
        self.avgdl = float(np.mean(list(self.doc_len.values()))) if self.N > 0 else 0.0

    # -------------------------- scoring -------------------------------
    def idf(self, term: str) -> float:
        df = self.df.get(term, 0)
        return math.log((self.N - df + 0.5) / (df + 0.5) + 1)

    def score(self, query_tokens: List[str], doc_id: str) -> float:
        doc_tf = self.doc_tf.get(doc_id, Counter())
        dl = self.doc_len.get(doc_id, 0)
        if dl == 0:
            return 0.0
        s = 0.0
        for term in query_tokens:
            f = doc_tf.get(term, 0)
            if f == 0:
                continue
            num = f * (self.k1 + 1)
            den = f + self.k1 * (1 - self.b + self.b * (dl / self.avgdl))
            s += self.idf(term) * (num / den)
        return s

    # -------------------------- retrieval -----------------------------
    def retrieve(self, query_text: str, k: int = 10) -> List[Tuple[str, float]]:
        q_tokens = self.tokenize(query_text)
        scores = []
        for doc_id in self.doc_tf.keys():
            s = self.score(q_tokens, doc_id)
            if s > 0:
                scores.append((doc_id, s))
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:k]

### Build the Cranfield Index

Load the Cranfield corpus with `ir_datasets` and call `bm25.build()`. You may use the best `k1` and `b` hyperparameters from Assignment 1.


In [5]:
# Load the Cranfield corpus and build the BM25 index
dataset = ir_datasets.load("cranfield")

# Best hyperparameters from Assignment 1 grid-search.
bm25 = BM25(k1=1.80, b=0.50)
bm25.build(dataset.docs_iter())

print(f"Index built: {bm25.N} documents, avg doc length = {bm25.avgdl:.1f} tokens")

Index built: 1400 documents, avg doc length = 102.9 tokens


### Task 1.2: RAGSystem

Implement `RAGSystem` with the following behaviour:

* **`build_prompt(query, retrieved_docs)`** — assemble the prompt using the **same instruction-tuning template** as in Assignment 2 (`format_prompt`). Concatenate the retrieved documents as a single doc to accomodate *k* larger than 1.
* **`generate(query, k, verbose)`** — full RAG pipeline:
  1. Retrieve top-*k* docs via `self.bm25.retrieve()`
  2. Build the prompt with `build_prompt`
  3. Run inference; **strip the prompt prefix** from the decoded output
  4. Return the answer string only
  5. Have a *verbose* flag to return the retrieved docs too. It will be useful for evaluation.
The constructor must accept `model`, `tokenizer`, `bm25`, `k` (default 3), and `max_new_tokens` (default 256).


In [6]:
class RAGSystem:
    """
    Retrieval-Augmented Generation (RAG) pipeline combining BM25 retrieval
    with a fine-tuned causal language model for open-form question answering.

    The pipeline operates in two stages:
      1. Retrieval: the BM25 index is queried to fetch the top-k most
         relevant documents for a given question.
      2. Generation: the retrieved documents are injected into an
         instruction-tuning prompt and passed to the language model, which
         produces a grounded answer.

    Typical usage:
        rag = RAGSystem(model=model, tokenizer=tokenizer, bm25=bm25)
        answer = rag.generate("What causes boundary layer separation?", k=3)
    """

    def __init__(
        self,
        model,
        tokenizer,
        bm25: BM25,
        max_new_tokens: int = 500,
    ):
        """
        Initialise the RAG system.

        Args:
            model          : fine-tuned causal language model used for generation.
            tokenizer      : tokenizer corresponding to *model*
            bm25           : a fully built BM25 instance (``bm25.build()`` must
                             have been called before passing it here).
            max_new_tokens : maximum number of new tokens the model may generate
                             per call to :meth:`generate`.
        """
        self.model = model
        self.tokenizer = tokenizer
        self.bm25 = bm25
        self.max_new_tokens = max_new_tokens
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)

    def build_prompt(
        self,
        query: str,
        retrieved_docs: list,
        instruction: str = None,
    ) -> str:
        """
        Assemble the instruction-tuning prompt used for RAG generation.

        Follows the same three-block template as Assignment 2's ``format_prompt``:
        ``### Instruction``, ``### Input``, ``### Response``.  

        Retrieved documents are concatenated (title + body) and inserted as the
        ``Document`` field of the input block, allowing the model to draw on
        multiple passages in a single forward pass.

        **Tip**: If you are unsure what the prompt should look like, inspect a
        few samples from your Assignment 2 test set — the structure used there
        is exactly what this method should replicate (minus the retrieved
        context, which replaces the original document field).

        Args:
            query         : the user question or instruction to be answered.
            retrieved_docs: list of document dicts, each containing ``"title"``
                            and ``"text"`` keys (as stored in ``BM25.docs_store``).
            instruction   : optional override for the ``### Instruction`` block.
                            Defaults to a generic technical-QA instruction when
                            *None*. Look at Test data samples for sample instructions

        Returns:
            Formatted prompt string ready for tokenisation, ending with
            ``"### Response:\n"`` so the model continues from that position.
        """
        if instruction is None:
            instruction = "Answer the following technical question using only the information in the document."

        # Concatenate retrieved passages (title + body) into a single Document
        # block, mirroring the "Question:\n...\n\nDocument:\n..." input layout
        # used during instruction-tuning in Assignment 2.
        doc_chunks = []
        for d in retrieved_docs:
            title = (d.get("title") or "").strip()
            text = (d.get("text") or "").strip()
            if title:
                doc_chunks.append(f"{title}\n{text}")
            else:
                doc_chunks.append(text)
        document_block = "\n\n".join(doc_chunks)

        input_text = f"Question:\n{query.strip()}\n\nDocument:\n{document_block}"
        prompt = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n"
        )
        return prompt

    def generate(
        self,
        query: str,
        k: int = 3,
        verbose: bool = False,
    ):
        """
        Run the full RAG pipeline for a single query.

        Steps:
          1. Retrieve the top-k documents from the BM25 index.
          2. Build the instruction-tuning prompt via ``build_prompt``.
          3. Tokenise the prompt and run generation with the language model.
          4. Decode only the newly generated tokens (the prompt prefix is stripped).
          5. Return the answer, optionally alongside the retrieved documents.

        Args:
            query   : raw (un-tokenised) user question.
            k       : number of documents to retrieve and include in the prompt.
            verbose : when *True*, returns a ``(answer, retrieved_docs)`` tuple
                      so callers can inspect which passages informed the answer.

        Returns:
            The generated answer string when verbose is False, or a
            ``(answer, retrieved_docs)`` tuple when verbose is True.
        """
        # 1. Retrieve top-k docs and gather their raw title/text records.
        hits = self.bm25.retrieve(query, k=k)
        retrieved_docs = [self.bm25.docs_store[doc_id] for doc_id, _ in hits]

        # 2. Build the instruction-tuning prompt.
        prompt = self.build_prompt(query, retrieved_docs)

        # 3. Tokenise and generate.
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        prompt_len = inputs["input_ids"].shape[1]
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                pad_token_id=self.tokenizer.eos_token_id,
                do_sample=False,
                repetition_penalty=1.15,
                no_repeat_ngram_size=4,
            )

        # 4. Decode only the newly generated tokens (drop the prompt prefix).
        answer = self.tokenizer.decode(
            output_ids[0][prompt_len:], skip_special_tokens=True
        ).strip()

        # 5. Truncate at the first prompt-format marker to stop repetition loops.
        answer = re.split(r'\n###', answer)[0].strip()

        if verbose:
            return answer, retrieved_docs

        return answer

### Task 1.3: Check your implementation

Run the cell below to verify that the full RAG pipeline works end-to-end.  
**Note:** this cell requires a loaded model and tokenizer — run it after you have loaded at least one checkpoint in Problem 2.

In [7]:
# Full RAG test
SAMPLE_QUERY = "what are the structural and aeroelastic problems associated with flight of high speed aircraft"

rag_test = RAGSystem(model=model_completion, tokenizer=tokenizer, bm25=bm25)

answer, docs = rag_test.generate(SAMPLE_QUERY, k=1, verbose=True)

print("=== Retrieved documents ===")
for i, doc in enumerate(docs, 1):
    title = doc.get("title", "").strip() or "(no title)"
    snippet = doc.get("text", "").replace("\n", " ")
    print(f"[{i}] {title}\n    {snippet}...\n")

print("=== Generated answer ===")
print(answer)

=== Retrieved documents ===
[1] some structural and aerelastic considerations of high
speed flight .
    some structural and aerelastic considerations of high speed flight .   the dominating factors in structural design of high-speed aircraft are thermal and aeroelastic in origin .  the subject matter is concerned largely with a discussion of these factors and their interrelation with one another .  a summary is presented of some of the analytical and experimental tools available to aeronautical engineers to meet the demands of high-speed flight upon aircraft structures .  the state of the art with respect to heat transfer from the boundary layer into the structure, modes of failure under combined load as well as thermal inputs and acrothermoelasticity is discussed .  methods of attacking and alleviating structural and aeroelastic problems of high-speed flight are summarized .  finally, some avenues of fundamental research are suggested ....

=== Generated answer ===
The given technica

---
# Tasks 2–4: Evaluation & Analysis

The remainder of this assignment is about evaluating and analysing the outputs of your RAG system by systematically varying inputs — such as queries, prompt instructions, and retrieval depth — and observing how the outputs change.

## Query Selection

<p style="font-size:1.2em">Manually select queries from the <strong>held-out test set</strong> of your Assignment 2 instruction-tuning data. Do <strong>not</strong> randomly sample from the full Cranfield dataset. If you modify or expand a query (e.g. for better retrieval), you may do so but must describe exactly how you did it. A single query may be reused across <strong>at most two</strong> tasks.</p>

Queries must come from the test set to avoid **data leakage** — if a query was seen during fine-tuning, the model may reproduce a memorised answer rather than reasoning from the retrieved context, making your evaluation unreliable.

---

## Task 2: Generator Evaluation

Compare your two fine-tuned models from Assignment 2 to choose the best generator for your system:
- **RAG A** — fine-tuned with loss over the **full prompt** (instruction + input + response)
- **RAG B** — fine-tuned with loss over the **completion only** (response tokens only)

**Note**: This task evaluates the generator in isolation. Both models receive the same retrieved context. Evaluate the generated responses only, not the quality of the retrieval.

### Implement RAG systems with both the models

In [8]:
# Instantiate one RAGSystem per model using the same BM25 index
rag_A = RAGSystem(model=model_full, tokenizer=tokenizer, bm25=bm25)
rag_B = RAGSystem(model=model_completion, tokenizer=tokenizer, bm25=bm25)

### Part 1: Select Queries

Choose queries covering **all three** conditions below. Fill in your final chosen queries in the code cell (TODO).

| Condition | Minimum | Your queries |
|-----------|---------|---------------|
| Simple factual query | 2 | |
| Query requiring a structured/formatted answer (e.g. bullet points) | 2 | |
| One query tested with two different reformulations (same keywords, different wording / instruction template) | 1 pair | |



In [10]:
K_TASK2 = 3

# Queries are taken from the held-out test set of the instruction-tuning
# dataset (test.jsonl in Assignment 2) so we know they were never seen
# during fine-tuning. Each entry is (label, instruction, query).
task2_queries = [
    # ---- Factual queries (>=2) ----
    (
        "factual-1 (test qid=218)",
        "Answer the following technical question using only the information in the document.",
        "what is the heat transfer to a blunt body in the absence of vorticity .",
    ),
    (
        "factual-2 (test qid=39)",
        "Answer the following technical question using only the information in the document.",
        "how can one detect transition phenomena in boundary layers .",
    ),
    # ---- Structured / formatted answer (>=2) ----
    (
        "structured-1 (test qid=50)",
        "Using only the document below, answer the question in bullet points.",
        "what are the effects of small amounts of gas rarefaction on the characteristics of the boundary layers on slender bodies .",
    ),
    (
        "structured-2 (test qid=137)",
        "Using only the document below, answer the question in bullet points.",
        "have any analytical studies been conducted on the time-to-failure mechanism associated with creep collapse for a long circular cylindrical shell .",
    ),
    # ---- Reformulation pair (same keywords, different wording + instruction) ----
    (
        "reform-v1 (test qid=67)",
        "Based on the document, define or describe the concept asked about in the question.",
        "can series expansions be found for the boundary layer on a flat plate in a shear flow .",
    ),
    (
        "reform-v2 (test qid=67, rephrased)",
        "Answer the following technical question using only the information in the document.",
        "describe how series expansions can be derived for the boundary layer of a flat plate immersed in a shear flow .",
    ),
]

for label, instruction, query in task2_queries:
    print("=" * 100)
    print(f"[{label}]")
    print(f"Instruction: {instruction}")
    print(f"Query: {query}")
    print("-" * 100)

    # Same retrieved context for both models -> isolates generator differences.
    hits = bm25.retrieve(query, k=K_TASK2)
    docs = [bm25.docs_store[doc_id] for doc_id, _ in hits]
    prompt = rag_A.build_prompt(query, docs, instruction=instruction)

    # Show retrieved documents 
    print("Retrieved documents (BM25, k={}):".format(K_TASK2))
    for rank, ((doc_id, score), doc) in enumerate(zip(hits, docs), start=1):
        title = doc.get("title", "").strip()
        text = doc.get("text", "").strip()
        print(f"  [{rank}] doc_id={doc_id}  score={score:.3f}  title={title}")
        print(f"      {text}")
    print("-" * 100)

    for name, rag in [("Full-prompt (RAG A)", rag_A), ("Completion-only (RAG B)", rag_B)]:
        inputs = rag.tokenizer(prompt, return_tensors="pt").to(rag.device)
        prompt_len = inputs["input_ids"].shape[1]
        with torch.no_grad():
            out = rag.model.generate(
                **inputs,
                max_new_tokens=rag.max_new_tokens,
                pad_token_id=rag.tokenizer.eos_token_id,
                do_sample=False,
                repetition_penalty=1.15,
                no_repeat_ngram_size=4,
            )
        answer = rag.tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()
        # Truncate at the first prompt-format marker to suppress fake
        # ### Explanation / ### Answer / ### Response continuation blocks
        # that the model emits after its first response.
        answer = re.split(r'\n###', answer)[0].strip()
        print(f"\n--- {name} ---\n{answer}")
    print()

[factual-1 (test qid=218)]
Instruction: Answer the following technical question using only the information in the document.
Query: what is the heat transfer to a blunt body in the absence of vorticity .
----------------------------------------------------------------------------------------------------
Retrieved documents (BM25, k=3):
  [1] doc_id=559  score=14.353  title=heat transfer at the forward stagnation point of blunt
bodies .
      heat transfer at the forward stagnation point of blunt
bodies .
  relations are presented for the calculation of heat transfer at
the forward stagnation point of both two-dimensional and axially
symmetric blunt bodies .  the relations for the heat transfer, which were
obtained from exact solutions to the equations of the laminar boundary
layer, are presented in terms of the local velocity gradient at the
stagnation point .  these exact solutions include effects of variation
of fluid properties, prandtl number, and transpiration cooling .
examples il

### Part 2: Evaluate the outputs of both models using the following criteria where applicable

Use the criteria below to evaluate your outputs and write down your observations.

| Criterion | Description |
|-----------|-------------|
| **Completeness** | Does the answer address the entire query given the available context? |
| **Instruction Following** | Does the model follow the required format / instructions? |
| **Robustness** | Is the model consistent across different reformulations of the same query? |
| **Fluency** | Is the output coherent and readable? |


#### Evaluation Table

| Query type | Model | Observation |
|------------|-------|-------------|
| Factual (qid=218 blunt-body heat transfer) | Full-prompt (RAG A) | **Ungrounded.** The output uses no terminology from the three retrieved abstracts (Fay & Riddell, local similarity, stagnation point, ratio formula) and instead produces a generic paragraph on *"numerical integration methods, finite element analysis, and spectral analysis"*, none of which appears in the retrieved documents. Completeness is low (the question is not answered); fluency is acceptable; the closing line about *"high-Reynolds-number flows… performance of hypersonic aircraft"* is not supported by any retrieved abstract. |
| | Completion-only (RAG B) | **Grounded paraphrase of document #3.** The response correctly invokes the *local similarity concept*, *Fay and Riddell*, and the *ratio of local heat-transfer rate to stagnation-point rate*. One factual error: the document states that local-similarity theory **underestimates** the actual rate by up to 25 %, whereas the response states that the theory overestimates it. No fabricated citations and no degenerate repetition. This is the best response in the batch on this query. |
| Factual (qid=39 transition phenomena) | RAG A | A short paragraph that reuses keywords from document #3 (insulated flat plate, skin-friction, turbulent boundary layer) but misreports a numerical value: the document gives the turbulent skin-friction coefficient as approximately 0.40, while the response renders it as *"effective Reynolds number between 4 and 5 x 10"*. The output then appends a spurious `## Conclusion:` header with a generic statement on hypersonic aircraft performance not present in any retrieved document. Only one detection method is described, so completeness is partial. |
| | RAG B | The only output in Task 2 that uses `-` bullet markers, although the instruction does not request them. The bullet content is largely grounded in document #3 (two-dimensional/local disturbances, air injection, floating-element/direct skin-friction measurements) but contains one factual error: *"turbulent skin-frict coefficient… approximately equal to zero"* (the document gives ~0.40). The tail of the response degenerates into repetition with token-level perturbations (`length-of 10mandwidth of1cm`). The structure is more organised than RAG A's, but the numerical error is more explicit. |
| Structured (qid=50 gas rarefaction, bullets) | RAG A | **No bullets.** The output enters degenerate repetition from the first line, cycling *"hypersonic viscous fluid past slender bodies"* through case- and spelling-level perturbations (`hyperonic`, `hypertonic`, `viscosellsell`, `viscosellsellliquid`). No grounded content is produced. This is the weakest response in the batch. |
| | RAG B | **No bullets.** The output also enters degenerate repetition (`hypersonic visco-liquids` → `viscual` → `VISCOUS FLUID PASSING SLUNGS OF REVISIONS`); the repeated span is shorter and remains closer to the original sentence than RAG A's. Same failure mode, marginally less severe. |
| Structured (qid=137 creep collapse, bullets) | RAG A | **No bullets.** A single run-on paragraph that introduces specifics absent from the retrieved documents: a *"temperature difference of 27 °C between outer and inner surface"* and a *"thickness less than diameter"* condition. The retrieved abstracts (variational theorem for creep, dead versus hydrostatic loading, aluminium-alloy cylinders at 300–500 °F) are not used. The response terminates with a sequence of unmatched parentheses (`) ) ) ) )`). |
| | RAG B | **No bullets.** The output drifts to generic *"cyclic process / cycles"* content (approximately 12 near-duplicate sentences on *"theoretical predictions for the time it takes for a cyclic cycle to complete"*). The retrieved abstracts concern creep collapse under sustained loading rather than cyclic loading, so this constitutes a topical hallucination. No fabricated numerical values, but also no grounded content. |
| Reform v1 vs v2 (qid=67) — v1 | RAG A | **Severe format failure.** The response begins a numbered list (*"1. The boundary layer consists of two layers…"*) and then produces a sequence of LaTeX environment commands: `\newcommand`, `\tilde`, `\begin{tabular}`, `\newcomb`, `\newcommand[0]{a}{a}`, and so on. No grounded content. |
| | RAG B | **Coherent format, ungrounded content.** The response is a well-formed 12-item numbered list, but the items describe generic geometry (*"outermost layer has constant thickness"*, *"innermost layer has constant height"*, *"middle layer"*, *"outer middle layer"*) and are unrelated to shear flow, series expansions, or the retrieved abstracts. The format is more interpretable than RAG A's, but the content is equally unsupported. |
| Reform v1 vs v2 (qid=67) — v2 | RAG A | **Best-grounded response in the batch.** A coherent paraphrase of document #2 (uniform suction, exact solution of the Navier–Stokes equations, infinite flat plate in a shear flow). No degenerate repetition, no LaTeX, no prompt regurgitation; the output terminates cleanly after a single paragraph. |
| | RAG B | A single generic sentence: *"The boundary layer in a simple shear flow… has been studied extensively… insights into the role of viscosity in determining the boundary layer thickness."* On topic, but with no specifics from any retrieved abstract. |

**Summary across criteria.**
- **Completeness:** Only RAG B on factual-1 and RAG A on reform-v2 actually answer the question with grounded content. Every other cell is either generic, off-topic, or degenerates into repetition.
- **Instruction Following:** This is the weakest dimension of the entire batch. The Task 2 prompts use four distinct `### Instruction` strings (factual-1/2 and reform-v2: *"answer using only the information in the document"*; structured-1/2: *"answer\u2026 in bullet points"*; reform-v1: *"define or describe the concept asked about"*). Behaviour against each:
    - **Structured / bullet-point instruction (structured-1, structured-2).** Neither model produces a single bullet on either query. Both ignore the format clause entirely. RAG A on structured-2 instead emits a parenthesis-only loop `) ) ) ) )`; RAG B on structured-2 emits twelve near-duplicate sentences about *"cyclic processes"*. The bullet-point instruction is the most consistently violated instruction in the batch.
    - **Spurious bullets on a non-structured query (factual-2).** RAG B emits `-`-prefixed bullet items on factual-2, where the instruction asks for a continuous-text answer. So the only place bullets appear in Task 2 is the one place the instruction did not ask for them \u2014 a clear sign that bullet output is uncorrelated with the instruction text and is instead triggered by something in the retrieved-doc surface form (doc #3 of factual-2 contains a list-style enumeration of experimental conditions).
    - **\"Define or describe the concept\" instruction (reform-v1).** Both models miss the framing. RAG A produces LaTeX (`\\newcommand`, `\\begin{tabular}`); RAG B produces a numbered list of generic geometry items (*\"outermost layer has constant thickness\"*) that defines no shear-flow concept and never references the retrieved abstracts.
    - **\"Use only the information in the document\" instruction (factual-1, factual-2, reform-v2).** Mixed. Only two of the six generations honour this clause: RAG B on factual-1 (uses doc #3's terminology) and RAG A on reform-v2 (paraphrases doc #2). The other four either invent off-document content (RAG A on factual-1: \"finite element analysis, spectral analysis\"), invert facts present in the document (RAG B on factual-2: skin-friction coefficient \"approximately equal to zero\" instead of ~0.40), or contradict the retrieved scope (RAG B on structured-2: drifts to \"cyclic processes\" when the abstracts are about sustained-load creep).
    - **Stop-when-done.** Both models routinely fail to stop after answering. RAG A's tails are LaTeX collapses or `## Conclusion:` headers; RAG B's tails are degenerate repetitions. Only one generation in the batch terminates cleanly after a single coherent answer (RAG A on reform-v2).
The structured / bullet-format instruction is fully ignored by both checkpoints; the *"only the document"* grounding instruction is honoured roughly 1 time in 3; and bullet markers, when they do appear, are triggered by document surface form rather than by the instruction. Format compliance does not differentiate the two models, but the *content-grounding* clause is honoured slightly more often by RAG B (factual-1) than by RAG A (reform-v2).
- **Robustness:** Both models are non-robust across the reform v1/v2 pair. RAG A swings between extremes (v1 = LaTeX collapse, v2 = best-grounded answer in the batch). RAG B swings between a fabricated 12-item geometry list (v1) and a one-sentence generic statement (v2). Neither is consistent.

- **Fluency:** Token-level fluency is acceptable in the opening lines for both models. The dominant failure mode in both is **degenerate repetition with token-level perturbations** at the tail of generation (`hyperonic → hypertonic → viscosellsell`; `length-of 10mandwidth of1cm`), not prompt regurgitation. RAG A also exhibits a LaTeX-environment collapse on reform-v1 that RAG B never produces.- **Hallucination / groundedness:** RAG A on factual-1 invents methods ("finite element analysis, spectral analysis") never mentioned in retrieval; RAG A on structured-2 invents a temperature-difference condition; RAG B on factual-1 inverts the sign of the local-similarity error (over- vs underestimate); RAG B on structured-2 drifts to "cyclic processes". Concrete numeric fabrications appear on both sides (RAG A: "27 °C", "4 to 5 × 10"; RAG B: "approximately equal to zero" for the skin-friction coefficient).

### Part 3: Based on your evaluations, answer the following questions

**Q1. Is there a query type where one model clearly outperforms the other?**

No query *type* shows a consistent advantage for either model across both of its instances. Two individual queries do, however, admit a clear ranking, and they favour opposite models.

- **factual-1 (qid=218, blunt-body heat transfer):** RAG B is preferable. Its response paraphrases retrieved document #3 and uses the relevant terminology (*local similarity*, *Fay and Riddell*, *ratio of local heat-transfer rate to stagnation-point rate*). It contains one factual error: it states that the local-similarity theory overestimates the actual rate, whereas the document states that it underestimates it by up to 25 %. RAG A on the same query references none of the retrieved abstracts and instead lists unrelated methods (numerical integration, finite element analysis, spectral analysis).
- **reform-v2 (qid=67, rephrased shear-flow query):** RAG A is preferable. It produces a coherent paraphrase of retrieved document #2 (uniform suction, exact Navier–Stokes solution for an infinite flat plate in a shear flow) and terminates after a single paragraph. RAG B returns one generic sentence that uses none of the retrieved content.

On the remaining queries both models fail, in different ways:

- factual-2: RAG A misreports the turbulent skin-friction coefficient and appends a spurious `## Conclusion:` header; RAG B uses bullet markers (which the instruction does not request) and reports the same coefficient as approximately zero.
- structured-1: both outputs enter degenerate repetition from the first line, with RAG A drifting further from the source sentence than RAG B.
- structured-2: neither output uses bullets; RAG A introduces a temperature-difference condition not present in the retrieved abstracts, and RAG B drifts to cyclic-process content unsupported by the (sustained-load) creep abstracts.
- reform-v1: RAG A produces uninterpretable LaTeX environment commands; RAG B produces a well-formed numbered list whose content is unrelated to shear flow.

The most consistent cross-query observation concerns format compliance rather than content: neither model produces bullet points on the two queries that request them, and the only bullet markers in the batch appear on a query that does not. Format compliance does not differentiate the two checkpoints.

**Q2. Which model do you select as the better generator, and what is your justification?**

I select the **completion-only model (RAG B)**. The margin is small and reflects relative rather than absolute quality.

- **Use of retrieved context on a factual query.** On factual-1, RAG B is the only model that reproduces terminology from the retrieved documents (Fay & Riddell, local similarity, stagnation-point ratio). RAG A produces content that does not appear in any retrieved abstract. Since the purpose of RAG is to exploit retrieved context, demonstrated use of that context is the primary criterion.
- **Severity of format failures.** On reform-v1, RAG A emits LaTeX environment commands (`\newcommand`, `\begin{tabular}`) that render the output. RAG B's failure on the same query is off-topic but well-formed.
- **Severity of degenerate repetition.** On structured-1, both models enter a degenerative loop, but RAG B's output remains closer to the source sentence than RAG A's.
- **Trade-offs.** RAG A produces the single best output in the batch (reform-v2) and a more grounded response than RAG B on structured-2. Selecting RAG B sacrifices these.

Neither model is adequate for production RAG at this scale (0.5B parameters). The selection is driven by RAG B's marginally more reliable use of retrieved context on the factual query and its less severe failure modes elsewhere. The most useful inference-time intervention for either checkpoint would be a stricter stopping criterion to truncate output before the degenerate repetition emerges.

**Q3. Does your selection align with the model you expected to perform better based on PPL_resp and PPL_all from Assignment 2? Discuss any discrepancy.**

The selection of RAG B is *consistent with* the prediction from PPL_resp but is not *driven* by it. Neither perplexity metric distinguishes the failure modes that dominate the qualitative evaluation, so perplexity alone would not have been a sufficient basis for the choice.

By construction, the completion-only model is expected to obtain a lower PPL_resp, since its training loss is computed exclusively on response tokens. The full-prompt model is correspondingly expected to obtain a lower PPL_all. For RAG only response quality matters at inference, so PPL_resp is the more relevant of the two signals, and the qualitative selection is consistent with it.

The discrepancies between the perplexity-based prediction and the observed behaviour are nevertheless substantial:

1. **PPL_resp does not predict the dominant failure modes.** Both models exhibit degenerate repetition at the tail of generation. Greedy decoding with a repetition penalty can drive a model into such loops while its average per-token loss on a held-out set remains low. Perplexity also does not measure isolated factual sign flips (RAG B's overestimates/underestimates) or LaTeX-environment hallucinations (RAG A on reform-v1). These phenomena dominate the qualitative evaluation and are not captured by either metric.
2. **PPL_resp does not predict instruction following.** RAG B produces bullet markers on the one query that does not request them and none on the two queries that do. Perplexity is computed token-by-token against a gold response and therefore cannot detect a mismatch between the format requested in the instruction and the format produced.
3. **The prompt-regurgitation prediction associated with PPL_all is not supported by these outputs.** A higher PPL_all for the completion-only model would suggest that the full-prompt model is more prone to re-emitting prompt scaffolding (`### Input:`, `Question:`, `Document:`) at inference. RAG A does not exhibit this behaviour on any query in the batch, including the reformulation pair. The PPL_all–PPL_resp gap is therefore a weaker selection signal than anticipated.

## Task 3: RAG System Evaluation

This task compares your selected model in two settings: with RAG and without retrieval (no-RAG). No-RAG serves as the baseline, and the model receives only the query with instruction and no retrieved context.

- **RAG**: top-*k* Cranfield documents are retrieved and injected into the prompt.
- **No-RAG**: the model answers from parametric knowledge only (no retrieved context).

The goal is to assess how much access to external context improves factual accuracy and reduces hallucination. Use the model you selected at the end of Task 2.

> **Note**: For RAG outputs, inspect the retrieved documents alongside the generated answer. You will need them to evaluate groundedness and use of context.

> **Important**: The no-RAG baseline must use the same `### Instruction` / `### Input` / `### Response` prompt template as the RAG setting, with the retrieved documents omitted. Passing the query as a bare string would turn it into a plain autocompletion prompt, which is not a fair comparison.

In [13]:
# Sample function for generating noRAG responses. You are allowed to edit as you wish 
def generate_no_rag(query: str, model, tokenizer, max_new_tokens: int = 256) -> str:
    """
    Generate an answer using the language model without any retrieved context.

    Serves as the no-RAG baseline: the model receives only the query wrapped
    in the standard instruction-tuning template and must rely entirely on its
    parametric (fine-tuned) knowledge to produce an answer.

    Args:
        query          : raw user question to answer.
        model          : causal language model to use for generation.
        tokenizer      : tokenizer corresponding to *model*.
        max_new_tokens : maximum number of new tokens to generate.

    Returns:
        Decoded answer string with the prompt prefix stripped and leading/
        trailing whitespace removed.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    prompt = (
        "### Instruction:\nAnswer the following technical question.\n\n"
        f"### Input:\nQuestion: {query}\n\n"
        "### Response:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )
    answer = tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True).strip()
    # Truncate at the first prompt-format marker to stop repetition loops.
    answer = re.split(r'\n###', answer)[0].strip()
    return answer


### Part 1: Select Queries

Select **at least 5** queries with at least one per condition. Avoid reusing the same set of queries used in Task 2.

| Condition | Description | Your query (TODO)|
|-----------|-------------|------------|
| Simple | Does not require specific domain knowledge | What is a wing? |
| Knowledge-intensive| Requires specific or detailed factual knowledge | what theoretical studies have been done on creep buckling of columns and plates . |
| Complex / multi-part | Concatenate two questions | how do interference-free longitudinal stability measurements made using free-flight models compare with similar measurements made in a wind tunnel, and what is the criterion for true panel flutter as opposed to small amplitude vibration arising from acoustic disturbances . |
| Comparative | Ask for similarities or differences between concepts | how do kuchemann's and multhopp's methods for calculating lift distributions on swept wings in subsonic flow compare with each other . |
| Out-of-domain | A random query outside the Cranfield aeronautics domain (e.g. What are the components of ibuprofen?) | what are the active components of ibuprofen . |

**Query modifications.** Three of the five queries are taken from the held-out Cranfield test split; the other two are intentionally synthetic to cover the *simple* and *out-of-domain* conditions. None of them overlap with the Task 2 query set. Specific edits:

- **Knowledge-intensive (test qid=132).** Test entry reads `"theoretical studies of creep buckling ."` (lines 15 / 40 / 58 / 159 of `test.jsonl`). I **expanded** it to `"what theoretical studies have been done on creep buckling of columns and plates ."` to turn the noun phrase into a well-formed question and to nudge BM25 toward documents that actually discuss columns and plates rather than generic theory.
- **Complex / multi-part (test qids 33 + 191).** This is a **concatenation of two test entries**, as the rubric for the complex condition requires:
  - First half from line 17: `"how do interference-free longitudinal stability measurements (made using free-flight models) compare with similar measurements made in a low-blockage wind tunnel ."` — I removed the parentheses and **simplified** `"low-blockage wind tunnel"` to `"a wind tunnel"` to keep the combined query readable.
  - Second half from line 19: `"what is the criterion for true panel flutter, as opposed to small amplitude vibration arising from acoustic disturbances ."` — used verbatim (joined with `", and "`).
- **Comparative (test qid=82).** Test entry (lines 33 / 41) reads `"how do kuchemann's and multhopp's methods for calculating lift distributions on swept wings in subsonic flow compare with each other and with experiment ."`. I **dropped the trailing `"and with experiment"`** clause so the query is a pure model-vs-model comparison and does not implicitly demand experimental data the retriever may not surface.
- **Simple (`"What is a wing?"`).** **Synthetic**, not from the test set. Chosen because it is a definitional question whose answer is common knowledge and does not require Cranfield-specific evidence — useful for probing whether retrieval helps, hurts, or is neutral on easy queries.
- **Out-of-domain (`"what are the active components of ibuprofen ."`).** **Synthetic**, not from the test set. Deliberately outside the Cranfield aeronautics domain so that BM25 is forced to return irrelevant aerodynamics documents and we can observe how the generator reacts to off-topic context.

In [14]:
# Task 3 — RAG vs no-RAG comparison.
# Based on Task 2 I picked the completion-only model as the generator.
model_chosen = model_completion
rag_system = RAGSystem(model=model_chosen, tokenizer=tokenizer, bm25=bm25)

K_TASK3 = 3

task3_queries = [
    ("simple",
     "What is a wing?"),
    ("knowledge-intensive (test qid=132)",
     "what theoretical studies have been done on creep buckling of columns and plates ."),
    ("complex / multi-part (test qid=33 + 191)",
     "how do interference-free longitudinal stability measurements made using free-flight models compare with similar measurements made in a wind tunnel, and what is the criterion for true panel flutter as opposed to small amplitude vibration arising from acoustic disturbances ."),
    ("comparative (test qid=82)",
     "how do kuchemann's and multhopp's methods for calculating lift distributions on swept wings in subsonic flow compare with each other ."),
    ("out-of-domain",
     "what are the active components of ibuprofen ."),
]

for label, query in task3_queries:
    print("=" * 100)
    print(f"[{label}]")
    print(f"Query: {query}")
    print("-" * 100)

    # ---- RAG ----
    answer_rag, docs = rag_system.generate(query, k=K_TASK3, verbose=True)
    print(f"\n--- RAG (k={K_TASK3}) retrieved documents ---")
    for i, d in enumerate(docs, 1):
        title = (d.get("title") or "(no title)").strip()
        snippet = (d.get("text") or "").replace("\n", " ")[:200]
        print(f"  [{i}] {title}\n      {snippet}...")
    print(f"\n--- RAG answer ---\n{answer_rag}")

    # ---- No-RAG baseline ----
    answer_no_rag = generate_no_rag(query, model_chosen, tokenizer)
    print(f"\n--- No-RAG answer ---\n{answer_no_rag}")
    print()

[simple]
Query: What is a wing?
----------------------------------------------------------------------------------------------------

--- RAG (k=3) retrieved documents ---
  [1] theoretical damping in roll and rolling moment due
to differential wing incidence for slender cruciform
wings and wing-body combinations .
      theoretical damping in roll and rolling moment due to differential wing incidence for slender cruciform wings and wing-body combinations .   a method of analysis based on slender-wing theory is develo...
  [2] application of two dimensional vortex theory to the
prediction of flow fields behind wings of wing-body
combinations at subsonic and supersonic speeds .
      application of two dimensional vortex theory to the prediction of flow fields behind wings of wing-body combinations at subsonic and supersonic speeds .   a theoretical investigation has been made of ...
  [3] a method for calculating the lift and centre of pressure
of wing-body-tail combinations at subsoni

### Part 2: Evaluate Outputs


| Criterion | Description |
|-----------|-------------|
| **Groundedness** | Is the answer supported and constrained by the retrieved context? |
| **Factual Accuracy** | Is the information correct? |
| **Hallucination** | Does the model introduce unsupported or incorrect information? |
| **Refusal** | Does the model acknowledge uncertainty or decline to answer? |
| **Use of Context** | Does the model effectively incorporate relevant retrieved details? |



#### Evaluation Table

| Query type | Setting | Observation |
|------------|---------|-------------|
| Simple (`"What is a wing?"`) | RAG | All three retrieved documents concern wing/body/tail aerodynamic combinations rather than a definition of a wing. The output paraphrases retrieved document #3 (*"The proposed method for calculating lift and center of pressure characteristics of circular cylindrical bodies in combination with triangular, rectangular, and trapezoid-shaped wings…"*) and then enters degenerate repetition (`"Lifted Are Expected To Be Within 1 %"` → `"Lifteds Were Expect To be inside"` → `"Effect Of Wings Flucting In Wing Tail Interruption"`). The question is not addressed; topic drift is severe. |
| | No-RAG | A short, coherent, partially correct response: *"A wing is a part of a bird's body that helps it fly. A wing is usually made of feathers, which are long, thin, and flexible…"*. The framing is restricted to biology and ignores aircraft wings, but the response contains no degenerate repetition and no fabricated specifics. |
| Knowledge-intensive (creep buckling, qid=132) | RAG | Retrieval is on-topic: all three hits are *"note on creep buckling of columns"*, covering Libove's initial-imperfection analysis, the variational theorem for creep, and the strain–time relation for a slightly crooked column. The generator does not use this content and instead produces a generic three-item list (*"The initial condition for the column is zero. The initial loading for the column has a value equal to one. The final condition for the columns is that its load equals the initial load."*) whose statements are not present in the retrieved documents. Use of context is minimal. |
| | No-RAG | Two contentless sentences: *"Theoretical studies have been done on creep buckling of columns and plates. These studies aim to understand the mechanisms…"*. No degenerate repetition, but no information either. |
| Complex / multi-part (qid=33 + 191) | RAG | Retrieval addresses only the first sub-question (three free-flight wind-tunnel-interference abstracts at M = 0.92–1.5); none of the documents discusses panel flutter or acoustic disturbances. The first sentence misreports *wind tunnel* as *wind turbine* (*"zero-lift drag on a wind turbine"*) and the response then enters degenerate repetition (`"Free-flight"` → `"Free-flower"` → `"Free-flow"` → `"fre flow"` → `"basif fre flow method"`). The second sub-question is never addressed. |
| | No-RAG | Confuses *free-flight* with *free-fall* and repeats the sentence *"The free-flight model is used to simulate the behavior of a free-fall model"* approximately fifteen times. Both sub-questions are ignored. |
| Comparative (Küchemann vs Multhopp, qid=82) | RAG | Retrieval is on-topic at the concept level (spanwise lift distributions on swept wings, modified strip analysis for flutter, end-plate effects), but none of the abstracts mentions Küchemann or Multhopp by name, since BM25 cannot match author keywords absent from the index. The output drifts to retrieved document #3 and produces three near-duplicate sentences on *"the proposed method of calculating the effect of endplate on swept wings"*. The Küchemann–Multhopp comparison is not attempted. The output terminates after three sentences without entering degenerate repetition. |
| | No-RAG | Produces two definitions, identical word-for-word, attributed in turn to *kuchemann's method* and *multhopp's method* (*"This method is based on the concept of 'lift' in the context of swept wings. The lift is calculated by taking the difference between the wing's center of gravity and the wing's tip…"*), followed by a contentless conclusion (*"both … compare with each other"*) and a chat-template breakage that emits an unrelated Chinese-language biology multiple-choice question. Both a high-confidence fabrication and a format failure. |
| Out-of-domain (ibuprofen) | RAG | BM25 returns chemically irrelevant aerospace abstracts (axial constraint on thin conical shells, turbulent boundary layer on ablating surfaces, heat conduction through a polyatomic gas). The output ignores both the question and the retrieved documents and describes *"an electric motor coupled to a hydraulic pump … extract water from the surrounding environment … turn on automatically during periods of high temperatures"*. The response is unrelated to ibuprofen and to anything in the retrieved documents. No degenerate repetition. |
| | No-RAG | Produces a numbered list of supposed active components — *Hydrochloride*, *Acetaminophen*, *Chloroquine*, *Phenylbutazone* — and then repeats the list. All four are incorrect: the active ingredient of ibuprofen is ibuprofen itself; *Acetaminophen* is the active ingredient of paracetamol/Tylenol, *Chloroquine* is an antimalarial, and *Phenylbutazone* is a different NSAID. The output is fluent and well formatted but factually wrong. |

**Main observations across the table.**

1. Retrieval is on-topic for three of the five queries (creep buckling directly, the free-flight half of the multi-part query directly, and swept-wing lift distributions at the concept level but not by author name). It is off-topic for the simple definition query and for the out-of-domain query, as expected by design.
2. On-topic retrieval does not translate into grounded responses. On the creep-buckling and Küchemann/Multhopp queries the generator largely ignores the retrieved context and produces fabricated or off-target content. The 0.5B generator is the limiting factor.
3. The dominant RAG failure mode is degenerate repetition with token-level perturbations rather than prompt regurgitation: greedy decoding on the completion-only checkpoint cycles the same sentence through case- and spelling-level variants until `max_new_tokens` is exhausted.
4. No-RAG hallucinations are typed (named entities, fabricated ingredient lists); RAG hallucinations are topical (drift into the retrieved jargon or into an unrelated narrative). The topical drift in RAG is harder for a non-expert to detect because the wording remains domain-related.
5. Neither setting refuses on the out-of-domain query. There is no relevance gate, so off-topic retrieval is treated as authoritative context.

### Part 3: Based on your evaluations, answer the following questions

**Q1. Overall, does RAG improve over no-RAG? Summarize your findings.**

In this run RAG does not improve over no-RAG on the majority of queries. The per-query ranking is:

| Query | Preferred setting | Reason |
|---|---|---|
| Simple ("What is a wing?") | No-RAG | The no-RAG response is a coherent, if biology-only, definition; the RAG response drifts into wing-body-tail aerodynamics and enters degenerate repetition. |
| Knowledge-intensive (creep buckling) | Tie (slight preference for no-RAG) | The no-RAG response is contentless filler; the RAG response retrieves the correct documents but the generator ignores them and fabricates *"initial condition zero, initial loading equal to one"*. Neither output is usable. |
| Complex / multi-part | Slight preference for RAG | Both responses fail. The RAG response begins on the correct sub-topic (free-flight measurements) before degenerating; the no-RAG response immediately confuses *free-flight* with *free-fall* and repeats the same sentence. |
| Comparative (Küchemann/Multhopp) | Slight preference for RAG | Both responses are wrong. The no-RAG response produces two identical definitions attributed to different authors and exhibits a chat-template breakage (Chinese MCQ leak); the RAG response only drifts to end-plate effects. Off-target paraphrase is preferable to confidently fabricated attributions. |
| Out-of-domain (ibuprofen) | Slight preference for RAG | Both responses are wrong. The no-RAG response fabricates a plausible-looking ingredient list using real drug names without any hedging or refusal; the RAG response drifts into an obviously off-topic narrative that any reader will reject on inspection. |

The 0.5B generator is too small to consistently exploit even on-topic retrieval; on off-topic retrieval RAG is harmful for the simple query but reduces the severity of the failure for the out-of-domain query.

**Q2. For which query types does RAG provide the most noticeable improvement, and why?**

The clearest, though still partial, RAG improvements are on the complex multi-part, comparative, and out-of-domain queries:

- **Complex multi-part.** RAG anchors the first sub-question to the actual free-flight wind-tunnel-interference abstracts, so the first sentence of the RAG response is at least topically correct. The no-RAG response substitutes *free-fall* for *free-flight* and never recovers from this confusion.
- **Comparative (Küchemann/Multhopp).** RAG does not attempt the comparison, but it does not fabricate identical author-attributed definitions in the way the no-RAG response does. Drifting to a related document is preferable to producing a fluent, plausible, copy-pasted falsehood under a named author.
- **Out-of-domain (ibuprofen).** When retrieval is off-topic, the RAG response drifts visibly enough that the failure is obvious; the no-RAG response produces a confident, well-formatted list of incorrect ingredients that a non-expert reader would believe.

The mechanism is the same in all three cases: the 0.5B base model has limited parametric knowledge of niche topics, so its un-grounded responses are either filler, fabrication, or confident plausible-sounding errors. Even partial topical anchoring — or, in the out-of-domain case, total anchoring to the wrong topic — prevents the most dangerous failure mode, which is confident fabrication of named entities.

There is no query in this batch on which RAG produces a fully correct, grounded response. "Improvement" here denotes a less dangerous failure mode rather than a useful answer.

**Q3. How much does retrieval affect hallucination? Does it consistently reduce hallucinations, or are there cases where it has little or no effect (or even introduces errors)? Discuss with examples.**

Retrieval does not consistently reduce hallucination; its effect depends on whether retrieval is on-topic and on whether the generator copies from it. Three patterns are observed:

- **No effect on hallucination, but a change in its type (creep buckling).** The retrieved abstracts contain real specifics (Libove's analysis, the variational theorem, the slightly-crooked-column strain–time relation). The generator does not copy any of them and instead produces three fabricated abstract conditions. Retrieval did not prevent the fabrication; the generator simply ignored the context.
- **Replacement of fabrication by topic drift (Küchemann/Multhopp; ibuprofen).** Without retrieval, the generator fabricates two identical author-attributed definitions on the comparative query and a plausible-looking fake ingredient list on the out-of-domain query. With retrieval, it drifts to end-plate effects in the first case and to an unrelated mechanical narrative in the second. The hallucinations did not disappear; their form changed from invented attributed claims (or invented entities) to off-target paraphrase. The latter is less dangerous because it does not produce a plausible-sounding fake list of drug ingredients.
- **Introduction of new hallucinations (simple "what is a wing?").** Retrieval forces the generator to discuss wing-body-tail aerodynamics that the no-RAG response would not have invoked. This is a hallucination caused by retrieval. The no-RAG biology answer, while incomplete, is not actively wrong.

Retrieval therefore acts as a redistributor of hallucination rather than as an eraser of it: it converts fabricated entities into off-target paraphrase. The direction depends entirely on retrieval quality and on whether the generator chooses to use the retrieved context.

**Q4. How does the overall quality of answers differ between RAG and no-RAG? Consider not just fluency but also substance and filler language.**

Both settings use the same generator, so token-level fluency is comparable. The substantive differences are:

- **Length and termination.** No-RAG responses are short and either coherent (simple, creep buckling) or repeat a single sentence (free-flight, ibuprofen). RAG responses are longer and tend to terminate via degenerate repetition at the tail (`Free-flight → Free-flower → fre flow`; `Lifted → Lifteds`). This failure mode does not occur in the no-RAG setting in this batch, most likely because the longer prompt (instruction plus three documents) shifts decoding into a regime in which greedy decoding with a repetition penalty drifts through near-duplicate tokens instead of halting.
- **Filler vs domain-related wording.** No-RAG responses are dominated by content-free filler (*"These studies aim to understand the mechanisms…"*, *"Both methods compare with each other"*). RAG responses are dominated by domain-related phrasing taken from the retrieved documents (*"aerodynamic influence coefficients"*, *"spanwise lift distributions"*, *"wind tunnel interference model"*), even when that phrasing does not address the question. RAG outputs appear more authoritative than their content warrants.
- **Failure type and severity.** No-RAG failures consist of inventing typed entities that look credible (incorrect ibuprofen ingredients drawn from the names of real drugs; identical definitions attributed to different authors; *free-fall* substituted for *free-flight*). These are the more dangerous failures because the responses are fluent, on-topic, and incorrect. RAG failures consist of drifting to a retrieved sub-topic or to an unrelated topic (end-plate effects in place of Küchemann/Multhopp; wing-body-tail combinations in place of a definition; electric motors in place of ibuprofen). Domain drift is easier for any reader to detect.
- **Format integrity.** The no-RAG response on the comparative query exhibits a chat-template breakage (an unrelated Chinese-language); no RAG response does so in this batch. The longer, more constrained RAG prompt appears to keep the generator on-format even when the content is incorrect.

**Q5. How does the model behave on the out-of-domain query with and without RAG? Which behavior is more desirable from a user's perspective? Explain why.**

The two behaviours on `"what are the active components of ibuprofen ."` are:

- **RAG.** BM25 returns three off-topic aerospace abstracts (axial constraint on thin conical shells, turbulent boundary layer on ablating surfaces, heat conduction through a polyatomic gas). The generator ignores both the question and the retrieved documents and produces a paragraph about *"an electric motor coupled to a hydraulic pump … extract water from the surrounding environment … programmed to turn on automatically during periods of high temperatures"*. The response is doubly ungrounded, neither about ibuprofen nor about the retrieved documents, but the failure is sufficiently obvious that no reader could overlook it.
- **No-RAG.** Produces a fluent, well-formatted numbered list of supposed active components — *Hydrochloride*, *Acetaminophen*, *Chloroquine*, *Phenylbutazone* — every entry incorrect, then repeats the list. The response remains in the pharmaceutical domain and resembles a real ingredient list.

The no-RAG response is the more dangerous of the two. It does not refuse, it does not hedge, and it returns a confident numbered list of real drug names that resembles a credible answer to the question, while every entry is incorrect. A non-expert reader has no signal that the response is fabricated and could act on it. The RAG response is more easily dismissed: a paragraph on electric motors and hydraulic pumps in answer to a question about a drug is sufficiently off-topic that any reader will reject it. Confident, fluent fabrication of plausible-looking facts, exactly what the no-RAG response does here — is the most damaging failure mode for a question-answering system, since it converts an *"I do not know"* situation into a well-formatted source of misinformation.

## Task 4: Error Analysis

Analyse when the RAG pipeline succeeds or fails, and identify where failures originate — retrieval, generation, or both.


### Part 1: Identify One Example per Case

Using open-form queries, find **at least one example** of each case below. For each case, 

(a) determine whether the root cause is retrieval, generation, or both,

(b) propose a solution for the error cases.

| Case | Description | Diagnostic questions |
|------|-------------|----------------------|
| **Refusal** | The model declines to answer or expresses uncertainty | Was the context relevant and sufficient? Did the model fail to use available information? |
| **Incorrect Answer** | The model provides a wrong or unsupported answer | Was the context relevant? Did the model correctly use the retrieved information? |
| **Correct Answer** | The model provides a correct and relevant answer | Was the context relevant? Did the model correctly use the retrieved information? |


In [15]:
K_TASK4 = 3

task4_queries = [
    ("Refusal slot — out-of-domain (system is expected NOT to refuse)",
     "what is the traditional recipe for italian tiramisu ."),
    ("Incorrect (test qid=96)",
     "has anyone investigated the unsteady lift distributions on finite wings in subsonic flow ."),
    ("Correct (test qid=15)",
     "theoretical studies of creep buckling ."),
]

for label, query in task4_queries:
    print("=" * 100)
    print(f"[{label}]")
    print(f"Query: {query}")
    print("-" * 100)

    # Print the raw BM25 scores so we can discuss retrieval quality
    # independently of generation.
    hits = bm25.retrieve(query, k=K_TASK4)
    print(f"BM25 top-{K_TASK4} scores: {[round(s, 3) for _, s in hits]}")

    answer, docs = rag_system.generate(query, k=K_TASK4, verbose=True)
    print(f"Retrieved documents (k={K_TASK4}):")
    for i, d in enumerate(docs, 1):
        title = (d.get("title") or "(no title)").strip()
        snippet = (d.get("text") or "").replace("\n", " ")[:200]
        print(f"  [{i}] {title}\n      {snippet}...")
    print(f"\n--- RAG answer ---\n{answer}\n")

[Refusal slot — out-of-domain (system is expected NOT to refuse)]
Query: what is the traditional recipe for italian tiramisu .
----------------------------------------------------------------------------------------------------
BM25 top-3 scores: [7.731]
Retrieved documents (k=3):
  [1] recent advances in the buckling of thin shells .
      recent advances in the buckling of thin shells . the importance of the field of shell analysis is evidenced by the fact that in august, 1959, the international union of theoretical and applied mechani...

--- RAG answer ---
The traditional recipe for Italian tiramisu consists of three ingredients: mascarpone cheese , sugar , and cocoa powder. The order of these ingredients can vary depending on personal preference or cultural background.

[Incorrect (test qid=96)]
Query: has anyone investigated the unsteady lift distributions on finite wings in subsonic flow .
------------------------------------------------------------------------------------------

#### Case Analysis

---

**Refusal case:**
- Query: *"what is the traditional recipe for italian tiramisu ."*
- Retrieved context relevant and sufficient: **No.** BM25 returned a single hit at every k≥1 (top-1 = 7.731, the lowest of the three queries) because no other Cranfield document shares any non-stopword token with the query. The single hit, *"recent advances in the buckling of thin shells"*, is a structural-mechanics abstract unrelated to the query.
- Generated answer: *"The traditional recipe for Italian tiramisu consists of three ingredients: mascarpone cheese, sugar, and cocoa powder. The order of these ingredients can vary depending on personal preference or cultural background."*
- Did the model fail to use available information: **Not applicable in the usual sense** — no relevant information was available. The generator bypassed the irrelevant `Document:` block entirely and answered from parametric knowledge. The model did not refuse and did not signal uncertainty. The answer is fluent and topical but factually under-specified.
- Root cause: **Both retrieval and generation.** Retrieval has no relevance threshold and forwarded a 7.7-scoring off-topic hit without flagging it; generation has no abstention behaviour and silently substitutes parametric knowledge when the retrieved passage is unusable. The failure differs from the ibuprofen probe in Task 3 (off-topic fabrication anchored to retrieved vocabulary): here the context is ignored entirely, producing a more plausible-looking and therefore more dangerous output.
- Proposed solution: (i) BM25-score threshold near 10 or a cross-encoder relevance check, with an explicit refusal branch when no document passes the threshold; (ii) instruction-tuning data containing abstention targets of the form *"the available documents do not address this question"*; (iii) decoding constraint requiring lexical overlap with the retrieved passages, suppressing parametric-knowledge fallback.

---

**Incorrect-answer case:**
- Query: *"has anyone investigated the unsteady lift distributions on finite wings in subsonic flow ."* 
- BM25 top-3 scores: **[24.262, 22.885, 22.004]** — the strongest cluster of the three queries.
- Retrieved context, all directly on-topic:
  1. *"an integral equation relating the general time-dependent lift and downwash distributions on finite wings in subsonic flow"* — answers the yes/no question on its own,
  2. *"the unsteady lift of a wing of finite aspect ratio"*,
  3. *"approximate indical lift functions for several wings of finite span in incompressible flow as obtained from oscillatory lift coefficients"*.
- Generated answer: *"The unsteady lift function for a wing with aspect ratio of 1/5 and height of 8 meters has been determined through numerical simulations. The unsteady lift functions for various types of wings have also been analyzed through numerical simulations."*
- Errors: (i) "aspect ratio of 1/5" and "height of 8 meters" appear in none of the retrieved abstracts; (ii) "numerical simulations" mis-states the methodology — the retrieved sources are analytical (integral equations, indicial functions, aerodynamic-inertia corrections); (iii) the yes/no question is never answered and no source is named.
- Context relevance: high. Context use: none.
- Root cause: **generator.** Retrieval was sufficient to answer the question; the generator did not copy from the context. Symmetric to the Refusal slot — both cases show parametric override of the retrieved passages, at opposite ends of the relevance spectrum.
- Proposed fix: (i) reduce *k* so a single high-scoring passage dominates the context window (tested in Part 2); (ii) per-passage delimiters in `build_prompt` with an explicit citation requirement; (iii) a larger generator with stronger copy-from-context behaviour.

---

**Correct-answer case:**
- Query: *"theoretical studies of creep buckling ."*
- BM25 top-3 scores: **[16.307, 15.403, 14.653]** — intermediate between the OOD baseline (7.7) and the qid=96 baseline (24+). The score level is driven by the rarity of "creep buckling" rather than by query length.
- Retrieved context, all directly on-topic:
  1. *"note on creep buckling of columns"* — Libove's analysis of initial imperfections in column buckling,
  2. *"note on creep buckling of columns"* (distinct paper) — theory and tests of creep collapse of long thin aluminium-alloy cylinders under external radial pressure under a steady-state creep assumption,
  3. *"comparison of theoretical and experimental creep buckling times of initially straight, centrally loaded columns"*.
- Generated answer: *"Theoretical studies of crepant buckling of long thin aluminum alloy cylinders under external radian pressure have been carried out recently. The purpose of these studies is twofold: (1) To understand how the initial loading conditions affect the ultimate buckling behavior of long thin aluminium alloy cylinders; (2) To develop new methods for predicting the ultimate buckbling behavior of long thick aluminium alloy cylinders."*
- Assessment: substantively correct. Every load-bearing noun phrase ("long thin aluminum alloy cylinders", "external ra(di)al pressure") is taken from doc [2]; no specifics are invented. Three residual defects: (a) token-level typos ("crepant", "radian", "buckbling"), consistent with `repetition_penalty=1.15` perturbing greedy decoding in a 0.5B model; (b) the "twofold purpose" enumeration is structural padding not present in the abstract; (c) docs [1] and [3] are not integrated, so the answer under-covers the retrieved set.
- Context relevance: high. Context use: faithful paraphrase of doc [2]; docs [1] and [3] ignored.
- Root cause of partial success: short query with a rare technical noun phrase, requiring single-passage compression rather than multi-document integration — the regime in which a 0.5B completion-only model is most reliable.
- Improvement levers: lower `repetition_penalty` (to suppress the token-level typos), an explicit single-sentence summary instruction (to suppress the invented enumeration), and a stronger generator (to integrate the remaining two passages).

---

**Cross-case observations.**
1. The system has no abstention mechanism. The Refusal slot is realised as a documented non-refusal in which the generator silently overrides the retrieved context with parametric knowledge — a fluent, topical, and factually incomplete output that is harder for a non-expert to detect than the off-topic fabrication observed on the ibuprofen probe in Task 3.
2. BM25 top-1 score correlates with retrieval relevance (7.7 OOD, 16.3 on-topic, 24.3 on-topic) but does not predict generator behaviour: the highest-scoring case (qid=96) produced the worst answer, while the middle-scoring case (qid=15) produced the most grounded one.
3. Generator behaviour partitions by integration load. Single-passage compression (qid=15) succeeds; multi-passage integration of analytical sources (qid=96) fails into fabricated specifics; absent or unusable retrieval (OOD) bypasses the context entirely. Retrieval quality is necessary but not sufficient.
4. The same decoding configuration (`repetition_penalty=1.15`, greedy) produces token-level typos in the Correct case and fabricated numerical specifics in the Incorrect case. Both are decoding-related artefacts and would be partially mitigated by a lower penalty and a stronger generator, independently of retrieval.

### Part 2: Experiment with at least two additional values of $k$

Select one query where retrieval was the cause of failure and one where the generator was the cause. 

Experiment at least two additional values of $k$ for each (one higher and one lower than your original setting) and discuss the results. 



In [16]:
retrieval_failure_query = "what is the traditional recipe for italian tiramisu ."

generator_failure_query = (
    "has anyone investigated the unsteady lift distributions on finite wings "
    "in subsonic flow ."
)

K_VALUES = [1, 3, 7]

for label, query in [
    ("Retrieval-caused failure (out-of-domain)", retrieval_failure_query),
    ("Generator-caused failure (test qid=96)", generator_failure_query),
]:
    print("=" * 100)
    print(f"[{label}]")
    print(f"Query: {query}")
    for k in K_VALUES:
        print("-" * 100)
        hits = bm25.retrieve(query, k=k)
        print(f"k = {k}  |  BM25 top-{k} scores: {[round(s, 3) for _, s in hits]}")
        answer = rag_system.generate(query, k=k)
        print(f"Answer (k={k}):\n{answer}\n")

[Retrieval-caused failure (out-of-domain)]
Query: what is the traditional recipe for italian tiramisu .
----------------------------------------------------------------------------------------------------
k = 1  |  BM25 top-1 scores: [7.731]
Answer (k=1):
The traditional recipe for Italian tiramisu consists of three ingredients: mascarpone cheese , sugar , and cocoa powder. The order of these ingredients can vary depending on personal preference or cultural background.

----------------------------------------------------------------------------------------------------
k = 3  |  BM25 top-3 scores: [7.731]
Answer (k=3):
The traditional recipe for Italian tiramisu consists of three ingredients: mascarpone cheese , sugar , and cocoa powder. The order of these ingredients can vary depending on personal preference or cultural background.

----------------------------------------------------------------------------------------------------
k = 7  |  BM25 top-7 scores: [7.731]
Answer (k=7):
Th

#### Discussion — effect of *k* on the two failure modes

Generator and BM25 index are held fixed; only *k ∈ {1, 3, 7}* varies. The system has no abstention mechanism: every value of *k* produces a retrieval and a generation.

---

**Retrieval-caused failure — out-of-domain query (*"what is the traditional recipe for italian tiramisu ."*).**

| k | BM25 scores | Answer |
|---|---|---|
| 1 | [7.731] | "The traditional recipe for Italian tiramisu consists of three ingredients: mascarpone cheese, sugar, and cocoa powder. The order of these ingredients can vary depending on personal preference or cultural background." |
| 3 | [7.731] | identical to k=1 |
| 7 | [7.731] | identical to k=1 |

- BM25 returns a single hit at every *k*: no other Cranfield document shares any non-stopword token with the query, so increasing *k* adds no passages.
- The generated answer is byte-for-byte identical for *k ∈ {1, 3, 7}*. Under greedy decoding, the generator disregards the retrieved abstract (an unrelated aerospace passage on structural buckling) and instead reproduces the same response from its parametric knowledge.
- *k* is inert in this regime. The structural fix is a relevance gate or a cross-encoder filter, with an explicit refusal branch.

---

**Generator-caused failure — qid=96 (*"has anyone investigated the unsteady lift distributions on finite wings in subsonic flow ."*).**

| k | BM25 scores | Failure mode |
|---|---|---|
| 1 | [24.262] | Meta-comment. *"The given text does not provide enough information to answer the provided technical question … there's no mention of pressure being involved at all. So … I cannot determine whether the given technical question has been answered correctly or incorrectly."* |
| 3 | [24.262, 22.885, 22.004] | Fabricated specifics. *"The unsteady lift function for a wing with aspect ratio of 1/5 and height of 8 meters has been determined through numerical simulations…"* |
| 7 | [24.262, 22.885, 22.004, 18.992, 18.746, 17.896, 16.535] | Repetition collapse. *"The results of the calculation of the dynamic responses of flexible lifting vehicles to random turbulence …"* — six near-identical sentence variants labelled (a)–(f), with progressive token degradation (*lifting* → *lifts* → *lift* → *luf*) until `max_new_tokens` is exhausted. |

- *k* = 1: the generator does not commit to a claim. The response is a meta-comment about insufficiency of the input rather than an answer to the yes/no question. No fabricated specifics are produced, but the retrieved abstract, which by itself answers the question affirmatively, is not used.
- *k* = 3: the failure mode of Part 1 reproduces. The generator fabricates "aspect ratio 1/5", "8 meters", and "numerical simulations", none of which appear in the three on-topic abstracts.
- *k* = 7: the prompt conditions on a lower-ranked passage on flexible-vehicle response to random turbulence (loosely related to unsteady lift) and enters a `repetition_penalty`-driven cycle that perturbs lexical items rather than terminating.
- *k* is non-monotone: meta-comment → confident fabrication → repetition collapse. No value of *k* produces a correct answer. The failure is generator-side and not addressable by a retrieval.

---

**Conclusion.**
1. *k* is inert when the corpus contains no relevant document: all three *k* values yield the identical parametric-knowledge response. Retrieval-bound failures require an explicitly refusal mechanism, not a higher *k*.
2. *k* alters the qualitative failure mode under high-quality retrieval (qid=96) but does not produce a correct answer at any setting. Generator-bound failures require a stronger generator, decoding constraints, or a citation-style prompt — not retrieval.
3. *k* = 1 is the safest setting for this generator across the two cases tested: identical to *k* ∈ {3, 7} on the OOD query, and the only setting on qid=96 that does not assert false specifics. The trade-off is reduced recall on multi-document queries not covered by this experiment.
4. The *k* = 7 collapse on qid=96 reflects an interaction between long context and `repetition_penalty=1.15`: once a sentence template is fixed, the penalty forces token-level perturbation (*lifting* → *lifts* → *lift* → *luf*) rather than termination. A lower penalty plus an early-stopping criterion would mitigate this independently of *k*.